06_survival.R — 对应论文 Figure 4C（生存分析）
论文做法：
  - 用 trajectory_score 的均值把患者分为 high / low
  - 在 ref (5,6) 发表的 bulk / 外部队列上做 Kaplan-Meier
  - 本仓库目前 **没有** ref (5,6) 的外部数据，本脚本只给出：
      1) 四个 EPN 样本细胞层面的 trajectory_score 分布 (补充结果)
      2) 留给用户的 TODO：下载 ref 5 (Gillen 2020 Cancer Cell) 和
         ref 6 (Gojo 2020 Cancer Cell) 补充表后，把患者 × score × survival
         合并到 surv_df；脚本下段的 survminer::ggsurvplot 即可出图
运行：conda run -n epn2_r Rscript projectmd/06_survival.R

In [ ]:
setwd('/home/zlcmc/wslproject')
source('workspace_paths.R')

In [ ]:
suppressPackageStartupMessages({
    library(survival); library(survminer); library(ggplot2); library(dplyr)
})

In [ ]:
SAMPLES <- c('GTE001','GTE002','GTE009','GTE012')
FIG6 <- output_path('fig4_survival')
dir.create(FIG6, showWarnings = FALSE, recursive = TRUE)

In [ ]:
# ---- 1) 合并四个样本的 trajectory_score 分布 ----
all_scores <- do.call(rbind, lapply(SAMPLES, function(s) {
    rds <- output_path(paste0('fig3/', s, '_scores.rds'))
    if (!file.exists(rds)) return(NULL)
    df <- readRDS(rds); df$sample <- s; df
}))
if (!is.null(all_scores) && nrow(all_scores) > 0) {
    p <- ggplot(all_scores, aes(x = sample, y = trajectory_score, fill = sample)) +
         geom_violin(trim = FALSE) + geom_boxplot(width = 0.1, fill = 'white') +
         theme_classic() +
         ggtitle('Trajectory score distribution (4 samples)')
    ggsave(file.path(FIG6, 'trajectory_score_per_sample.pdf'),
           p, width = 6, height = 5)
    cat('已输出:', file.path(FIG6, 'trajectory_score_per_sample.pdf'), '\n')
}

In [ ]:
# ---- 2) 外部生存队列（模板代码）----
cat('\n----- 外部生存分析（模板） -----\n')
cat('需要用户自行下载:\n')
cat('  ref (5) Gillen 2020 Cancer Cell — PF-EPN 患者 metadata + OS\n')
cat('  ref (6) Gojo 2020 Cancer Cell   — EPN 患者 metadata + OS\n')
cat('把 patient × trajectory_score × time × event 合成 surv_df 后，本段代码:\n\n')

In [ ]:
cat(paste(readLines(textConnection("
# surv_df <- read.csv(raw_data_path('external_EPN_cohort_with_scores.csv'))
# surv_df$group <- ifelse(surv_df$trajectory_score >= mean(surv_df$trajectory_score),
#                         'High', 'Low')
# fit <- survfit(Surv(OS_time, OS_event) ~ group, data = surv_df)
# p_km <- ggsurvplot(fit, data = surv_df, pval = TRUE, risk.table = TRUE,
#                    palette = c('firebrick','steelblue'),
#                    title = 'Trajectory-high vs -low OS')
# pdf(file.path(FIG6, 'Fig4C_KM_trajectory.pdf'), width = 7, height = 8)
# print(p_km); dev.off()
")), collapse = '\n'))
cat('\n\n✓ 06_survival.R 占位完成（等外部数据）\n')